In [0]:
# ============================================================
# MVP VTI CBL
# Notebook: 05_Analises
#
# Objetivo:
# Analisar os resultados produzidos pela camada Gold
# e responder às perguntas de negócio do MVP.
#
# P1 - Blocos com maior criticidade relativa de CBL
# P2 - Blocos com maior tendência de aumento do CBL
# P3 - Evolução global do CBL ao longo do tempo
# P4 - Locais/taxonomias com maiores valores de CBL
#
# Observação:
# A criticidade utilizada neste MVP é relativa à população
# analisada e não representa um limite técnico de manutenção.
# ============================================================

In [0]:
# Importa as funções do PySpark utilizadas nas agregações, filtros e análises dos resultados da camada Gold
from pyspark.sql import functions as F

In [0]:
# Carrega a tabela fato com as medições CBL integradas aos Locais e Blocos de 25 m
df_fato = spark.table("mvp_vti.gold.fato_cbl_bloco")
# Carrega as agregações Gold utilizadas nas análises de criticidade e evolução temporal
df_bloco = spark.table("mvp_vti.gold.agg_cbl_bloco")
df_dia = spark.table("mvp_vti.gold.agg_cbl_bloco_dia")
df_semana = spark.table("mvp_vti.gold.agg_cbl_bloco_semana")
# Carrega os indicadores de tendência temporal calculados para cada Bloco de 25 m
df_degradacao = spark.table("mvp_vti.gold.agg_degradacao_bloco")

In [0]:
# Confere o volume das principais tabelas Gold utilizadas nas análises finais do MVP
# As tabelas possuem granularidades diferentes, portanto suas contagens representam produtos analíticos distintos
print("=== TABELAS GOLD UTILIZADAS NAS ANÁLISES ===")
print("Fato CBL:", df_fato.count())
print("Agregação por bloco:", df_bloco.count())
print("Agregação diária:", df_dia.count())
print("Agregação semanal:", df_semana.count())
print("Tendência por bloco:", df_degradacao.count())

=== TABELAS GOLD UTILIZADAS NAS ANÁLISES ===
Fato CBL: 42881
Agregação por bloco: 7709
Agregação diária: 40577
Agregação semanal: 35369
Tendência por bloco: 2972


In [0]:
# Organiza os Blocos de 25 m pelo CBL P95 observado durante o período analisado
# O P95 é utilizado como referência para comparar a criticidade relativa entre os segmentos
df_ranking_blocos = (
df_bloco
.select(
"linha",
"bloco_id",
"bloco_inicio_m",
"bloco_fim_m",
"qtd_medicoes",
"cbl_medio",
"cbl_max",
"cbl_p95",
"cbl_std"
)
.orderBy(F.desc("cbl_p95"))
)
# Exibe os 20 blocos com maiores valores de CBL P95
display(df_ranking_blocos.limit(20))

linha,bloco_id,bloco_inicio_m,bloco_fim_m,qtd_medicoes,cbl_medio,cbl_max,cbl_p95,cbl_std
LP002,LP002_809450_809475,809450,809475,38,0.15184210526315792,0.35,0.33,0.06194397046145139
LP002,LP002_858175_858200,858175,858200,62,0.22322580645161294,0.41,0.33,0.06367886113849987
LP002,LP002_551950_551975,551950,551975,24,0.16,0.32,0.32,0.06587207629020517
LP002,LP002_323625_323650,323625,323650,26,0.2034615384615384,0.31,0.31,0.05775412073210414
LP002,LP002_114050_114075,114050,114075,15,0.14666666666666667,0.31,0.31,0.056146068004633704
LP001,LP001_145900_145925,145900,145925,2,0.21,0.3,0.3,0.12727922061357855
LP001,LP001_244850_244875,244850,244875,15,0.1753333333333333,0.3,0.3,0.055916604570258606
LP001,LP001_830250_830275,830250,830275,4,0.1775,0.3,0.3,0.08381527307120104
LP002,LP002_261800_261825,261800,261825,12,0.1975,0.3,0.3,0.07212678231609064
LP002,LP002_338975_339000,338975,339000,17,0.16117647058823525,0.3,0.3,0.05743282521141291


In [0]:
# Calcula os percentis de referência do CBL P95 para classificar os blocos de forma relativa
p50, p80 = df_ranking_blocos.approxQuantile(
"cbl_p95",
[0.50, 0.80],
0.01
)
# Classifica os blocos conforme sua posição na distribuição do CBL P95
# A classificação é relativa à população analisada e não representa um limite técnico de manutenção
df_ranking_blocos = (
df_ranking_blocos
.withColumn(
"criticidade_relativa",
F.when(F.col("cbl_p95") >= p80, "ALTA")
.when(F.col("cbl_p95") >= p50, "MEDIA")
.otherwise("BAIXA")
)
)
# Exibe os 20 blocos com maiores valores de CBL P95 e sua classificação relativa
display(
df_ranking_blocos
.select(
"linha",
"bloco_id",
"bloco_inicio_m",
"bloco_fim_m",
"qtd_medicoes",
"cbl_medio",
"cbl_max",
"cbl_p95",
"criticidade_relativa"
)
.orderBy(F.desc("cbl_p95"))
.limit(20)
)

linha,bloco_id,bloco_inicio_m,bloco_fim_m,qtd_medicoes,cbl_medio,cbl_max,cbl_p95,criticidade_relativa
LP002,LP002_809450_809475,809450,809475,38,0.15184210526315792,0.35,0.33,ALTA
LP002,LP002_858175_858200,858175,858200,62,0.22322580645161294,0.41,0.33,ALTA
LP002,LP002_551950_551975,551950,551975,24,0.16,0.32,0.32,ALTA
LP002,LP002_323625_323650,323625,323650,26,0.2034615384615384,0.31,0.31,ALTA
LP002,LP002_114050_114075,114050,114075,15,0.14666666666666667,0.31,0.31,ALTA
LP001,LP001_145900_145925,145900,145925,2,0.21,0.3,0.3,ALTA
LP001,LP001_244850_244875,244850,244875,15,0.1753333333333333,0.3,0.3,ALTA
LP001,LP001_830250_830275,830250,830275,4,0.1775,0.3,0.3,ALTA
LP002,LP002_261800_261825,261800,261825,12,0.1975,0.3,0.3,ALTA
LP002,LP002_338975_339000,338975,339000,17,0.16117647058823525,0.3,0.3,ALTA


Databricks visualization. Run in Databricks to view.

In [0]:
# Mantém somente os blocos que possuem tendência temporal do CBL P95 calculada
# Em seguida, classifica a direção da tendência conforme o sinal do slope
df_ranking_tendencia = (
df_degradacao
.filter(F.col("slope_cbl_p95").isNotNull())
.withColumn(
"tendencia",
F.when(F.col("slope_cbl_p95") > 0, "CRESCENTE")
.when(F.col("slope_cbl_p95") < 0, "DECRESCENTE")
.otherwise("ESTAVEL")
)
.orderBy(F.desc("slope_cbl_p95"))
)
# Apresenta a distribuição dos blocos conforme a direção da tendência observada
display(
df_ranking_tendencia
.groupBy("tendencia")
.count()
.orderBy(F.desc("count"))
)

tendencia,count
CRESCENTE,1634
DECRESCENTE,1312
ESTAVEL,26


In [0]:
# Seleciona somente os blocos que apresentam tendência positiva do CBL P95
# A ordenação destaca os segmentos com maior crescimento relativo do indicador ao longo do período analisado
df_top20_tendencia = (
df_ranking_tendencia
.filter(F.col("slope_cbl_p95") > 0)
.select(
"linha",
"bloco_id",
"bloco_inicio_m",
"bloco_fim_m",
"qtd_semanas",
"cbl_medio_periodo",
"cbl_max_periodo",
"cbl_p95_max_periodo",
"slope_cbl_medio",
"slope_cbl_p95",
"tendencia"
)
.orderBy(F.desc("slope_cbl_p95"))
.limit(20)
)
# Exibe os 20 blocos com maior tendência positiva do CBL P95
display(df_top20_tendencia)

linha,bloco_id,bloco_inicio_m,bloco_fim_m,qtd_semanas,cbl_medio_periodo,cbl_max_periodo,cbl_p95_max_periodo,slope_cbl_medio,slope_cbl_p95,tendencia
LP002,LP002_518350_518375,518350,518375,4,0.17875000000000002,0.25,0.25,0.003430232558139593,0.006007751937984581,CRESCENTE
LP002,LP002_842000_842025,842000,842025,4,0.1375,0.17,0.17,0.0015789473684210526,0.005526315789473685,CRESCENTE
LP002,LP002_800525_800550,800525,800550,4,0.1225,0.14,0.14,0.005384615384615269,0.005384615384615269,CRESCENTE
SD001,SD001_65750_65775,65750,65775,4,0.11249999999999999,0.13,0.13,0.0025714285714285696,0.005142857142857144,CRESCENTE
SD002,SD002_73625_73650,73625,73650,5,0.162,0.23,0.23,0.005074626865671654,0.005074626865671654,CRESCENTE
LP002,LP002_712575_712600,712575,712600,6,0.14666666666666667,0.2,0.2,0.005030927835051633,0.005030927835051633,CRESCENTE
LP002,LP002_659075_659100,659075,659100,7,0.13714285714285715,0.21,0.21,0.004749103942652484,0.004749103942652484,CRESCENTE
LP002,LP002_296075_296100,296075,296100,16,0.16358333333333333,0.36,0.36,0.002281357730045105,0.0044729514717582,CRESCENTE
LP002,LP002_636675_636700,636675,636700,4,0.1375,0.18,0.18,0.00492957746478868,0.0043380281690140205,CRESCENTE
LP002,LP002_742025_742050,742025,742050,4,0.1275,0.15,0.15,0.004333333333333249,0.004333333333333249,CRESCENTE


Databricks visualization. Run in Databricks to view.

In [0]:
# Considera somente medições com CBL preenchido para evitar distorções na evolução temporal
# A agregação semanal reduz oscilações diárias e permite observar o comportamento global do indicador
df_evolucao_global = (
df_fato
.filter(F.col("cbl").isNotNull())
.withColumn(
"semana",
F.date_trunc("week", F.col("date_time"))
)
.groupBy("semana")
.agg(
# Quantidade de medições disponíveis em cada semana
F.count("*").alias("qtd_medicoes"),
# Indicadores utilizados para acompanhar o comportamento semanal do CBL
F.avg("cbl").alias("cbl_medio"),
F.expr("percentile_approx(cbl, 0.50)").alias("cbl_p50"),
F.expr("percentile_approx(cbl, 0.95)").alias("cbl_p95"),
F.max("cbl").alias("cbl_max")
)
.orderBy("semana")
)
# Exibe a evolução semanal consolidada do CBL para toda a malha analisada
display(df_evolucao_global)

semana,qtd_medicoes,cbl_medio,cbl_p50,cbl_p95,cbl_max
2025-03-10T00:00:00.000Z,512,0.13386718749999985,0.12,0.19,0.27
2025-03-17T00:00:00.000Z,301,0.12840531561461768,0.12,0.17,0.28
2025-03-24T00:00:00.000Z,605,0.12728925619834697,0.12,0.17,0.23
2025-03-31T00:00:00.000Z,368,0.12831521739130414,0.12,0.18,0.22
2025-04-07T00:00:00.000Z,205,0.1345853658536585,0.12,0.19,0.27
2025-04-14T00:00:00.000Z,933,0.12943193997856403,0.12,0.18,0.27
2025-04-21T00:00:00.000Z,580,0.13048275862068953,0.12,0.18,0.33
2025-08-18T00:00:00.000Z,213,0.1354460093896713,0.13,0.19,0.23
2025-08-25T00:00:00.000Z,969,0.1340454076367393,0.12,0.19,0.27
2025-09-01T00:00:00.000Z,1209,0.13269644334160563,0.12,0.19,0.33


Databricks visualization. Run in Databricks to view.